<a href="https://colab.research.google.com/github/tonHS/Canadian-Crime-Trends/blob/Sandbox/Copy_Crime_Stats_MVP.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:

# Configure Git with your info
!git config --global user.name "tonHS"
!git config --global user.email "171510635+tonHS@users.noreply.github.com"

# Store credentials so you don't have to enter token every time
!git config --global credential.helper store

In [ ]:
# Install Dependencies and Packages
!pip install stats-can openpyxl

import pandas as pd
import requests
import zipfile
from io import BytesIO
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
from datetime import datetime


In [ ]:
# Fetch organized crime data
# Create directories

data_dir = Path('data')
data_dir.mkdir(exist_ok=True)
outputs_dir = Path('outputs')
outputs_dir.mkdir(exist_ok=True)

print("=" * 80)
print("FETCHING ORGANIZED CRIME DATA FROM STATISTICS CANADA")
print("=" * 80)

# StatCan table ID for organized crime
TABLE_ID = "35100062"

# Construct the download URL (using the direct ZIP download link)
download_url = f"https://www150.statcan.gc.ca/n1/tbl/csv/{TABLE_ID}-eng.zip"

try:
    print(f"\n📥 Downloading data from Statistics Canada (Table {TABLE_ID})...")
    print(f"URL: {download_url[:80]}...")

    # Download the data
    response = requests.get(download_url, timeout=30)
    response.raise_for_status()

    # Extract ZIP file
    with zipfile.ZipFile(BytesIO(response.content)) as zip_file:
        # Get the CSV file name (should be the table ID)
        csv_files = [f for f in zip_file.namelist() if f.endswith('.csv')]

        if not csv_files:
            raise ValueError("No CSV file found in downloaded ZIP")

        csv_filename = csv_files[0]
        print(f"✓ Downloaded and extracted: {csv_filename}")

        # Read the CSV
        with zip_file.open(csv_filename) as csv_file:
            df = pd.read_csv(csv_file)

    print(f"✓ Data loaded successfully: {len(df):,} rows, {len(df.columns)} columns")

    # Save raw data
    raw_data_path = data_dir / 'organized_crime_raw.csv'
    df.to_csv(raw_data_path, index=False)
    print(f"✓ Raw data saved to: {raw_data_path}")

    # Display basic info
    print(f"\n📊 Dataset Overview:")
    print(f"   • Time period: {df['REF_DATE'].min()} to {df['REF_DATE'].max()}")
    print(f"   • Columns: {', '.join(df.columns.tolist())}")

except requests.exceptions.RequestException as e:
    print(f"❌ Error downloading data: {e}")
    raise
except Exception as e:
    print(f"❌ Error processing data: {e}")
    raise

print("\n" + "=" * 80)
print("✓ DATA FETCH COMPLETE")
print("=" * 80)

FETCHING ORGANIZED CRIME DATA FROM STATISTICS CANADA

📥 Downloading data from Statistics Canada (Table 35100062)...
URL: https://www150.statcan.gc.ca/n1/tbl/csv/35100062-eng.zip...
✓ Downloaded and extracted: 35100062.csv
✓ Data loaded successfully: 702 rows, 15 columns
✓ Raw data saved to: data/organized_crime_raw.csv

📊 Dataset Overview:
   • Time period: 2016 to 2024
   • Columns: REF_DATE, GEO, DGUID, Most serious violation, UOM, UOM_ID, SCALAR_FACTOR, SCALAR_ID, VECTOR, COORDINATE, VALUE, STATUS, SYMBOL, TERMINATED, DECIMALS

✓ DATA FETCH COMPLETE


In [ ]:
# Organized Crime - Process and Clean Data
print("=" * 80)
print("PROCESSING ORGANIZED CRIME DATA")
print("=" * 80)

# Display the structure
print(f"\n📋 Data Structure:")
print(df.head())
print(f"\n📊 Column Types:")
print(df.dtypes)

# Clean and prepare the data
# Filter for actual crime violations (not totals in the violation name)
df_clean = df.copy()

# Convert REF_DATE to integer year
df_clean['Year'] = df_clean['REF_DATE'].astype(int)

# Filter for 2016-2024 only
df_clean = df_clean[df_clean['Year'].between(2016, 2024)]

# Get the value column (usually 'VALUE')
value_col = 'VALUE'
if value_col not in df_clean.columns:
    # Try to find the value column
    possible_cols = [col for col in df_clean.columns if 'value' in col.lower()]
    if possible_cols:
        value_col = possible_cols[0]
    else:
        raise ValueError("Could not find value column in data")

print(f"\n✓ Using '{value_col}' as the value column")

# Remove any rows with missing values
df_clean = df_clean[df_clean[value_col].notna()]

# Get violation column name
violation_col = 'Violations'
if violation_col not in df_clean.columns:
    possible_cols = [col for col in df_clean.columns if 'violation' in col.lower()]
    if possible_cols:
        violation_col = possible_cols[0]
    else:
        raise ValueError("Could not find violation column in data")

print(f"✓ Using '{violation_col}' as the violation column")

# Display unique violations
print(f"\n🔍 Unique violations found: {df_clean[violation_col].nunique()}")
print("\nSample violations:")
for i, violation in enumerate(df_clean[violation_col].unique()[:10], 1):
    print(f"   {i}. {violation}")

# Filter out "Total" rows to get specific violations
df_violations = df_clean[~df_clean[violation_col].str.contains('Total', case=False, na=False)]

print(f"\n✓ Filtered to {len(df_violations):,} rows of specific violations")
print(f"✓ Years covered: {df_violations['Year'].min()} to {df_violations['Year'].max()}")
print(f"✓ Number of specific violation types: {df_violations[violation_col].nunique()}")

print("\n" + "=" * 80)
print("✓ DATA PROCESSING COMPLETE")
print("=" * 80)

PROCESSING ORGANIZED CRIME DATA

📋 Data Structure:
   REF_DATE                               GEO           DGUID  \
0      2016  Canada, selected police services  2021A000011124   
1      2016  Canada, selected police services  2021A000011124   
2      2016  Canada, selected police services  2021A000011124   
3      2016  Canada, selected police services  2021A000011124   
4      2016  Canada, selected police services  2021A000011124   

                              Most serious violation     UOM  UOM_ID  \
0                              Total, all violations  Number     223   
1                                          Homicide   Number     223   
2       Attempted murder / Conspire to commit murder  Number     223   
3                                            Assault  Number     223   
4  Sexual violations (not including against child...  Number     223   

  SCALAR_FACTOR  SCALAR_ID       VECTOR  COORDINATE  VALUE  STATUS  SYMBOL  \
0         units          0  v1066366742        

In [ ]:
# Create Organized Crime Table - Top 20 Crimes in 2024 with Growth Metrics
print("=" * 80)
print("CREATING TOP 20 CRIMES TABLE")
print("=" * 80)

# Get 2024 data
df_2024 = df_violations[df_violations['Year'] == 2024].copy()

# Get 2019 data
df_2019 = df_violations[df_violations['Year'] == 2019].copy()

# Get 2016 data
df_2016 = df_violations[df_violations['Year'] == 2016].copy()

# Calculate totals for each year by violation type
violations_2024 = df_2024.groupby(violation_col)[value_col].sum()
violations_2019 = df_2019.groupby(violation_col)[value_col].sum()
violations_2016 = df_2016.groupby(violation_col)[value_col].sum()

# Get top 20 crimes by 2024 violations
top_20_violations = violations_2024.nlargest(20)

# Create the summary table
summary_data = []

for violation in top_20_violations.index:
    count_2024 = violations_2024.get(violation, 0)
    count_2019 = violations_2019.get(violation, 0)
    count_2016 = violations_2016.get(violation, 0)

    # Calculate growth
    if count_2019 > 0:
        growth_2019_2024 = ((count_2024 - count_2019) / count_2019) * 100
    else:
        growth_2019_2024 = float('inf') if count_2024 > 0 else 0

    if count_2016 > 0:
        growth_2016_2024 = ((count_2024 - count_2016) / count_2016) * 100
    else:
        growth_2016_2024 = float('inf') if count_2024 > 0 else 0

    summary_data.append({
        'Rank': len(summary_data) + 1,
        'Violation Type': violation,
        '2024 Violations': int(count_2024),
        'Growth 2019-2024 (%)': growth_2019_2024,
        'Growth 2016-2024 (%)': growth_2016_2024
    })

# Create DataFrame
df_summary = pd.DataFrame(summary_data)

# Format the table for display
df_display = df_summary.copy()
df_display['2024 Violations'] = df_display['2024 Violations'].apply(lambda x: f'{x:,}')
df_display['Growth 2019-2024 (%)'] = df_display['Growth 2019-2024 (%)'].apply(
    lambda x: 'N/A' if x == float('inf') else f'{x:+.1f}%'
)
df_display['Growth 2016-2024 (%)'] = df_display['Growth 2016-2024 (%)'].apply(
    lambda x: 'N/A' if x == float('inf') else f'{x:+.1f}%'
)

print(f"\n📊 TOP 20 ORGANIZED CRIMES IN CANADA (2024)")
print("=" * 80)
print(df_display.to_string(index=False))
print("=" * 80)

# Save to CSV
csv_path = outputs_dir / 'top_20_crimes_2024_with_growth.csv'
df_summary.to_csv(csv_path, index=False)
print(f"\n✓ Table saved to: {csv_path}")

# Create a nicely formatted display table
from IPython.display import display, HTML

# Style the dataframe for better display
styled_df = df_display.style.set_properties(**{
    'text-align': 'left',
    'font-size': '12px',
    'border-color': 'black'
}).set_table_styles([
    {'selector': 'th', 'props': [
        ('background-color', '#2E86AB'),
        ('color', 'white'),
        ('font-weight', 'bold'),
        ('text-align', 'center'),
        ('padding', '10px')
    ]},
    {'selector': 'td', 'props': [
        ('padding', '8px'),
        ('border', '1px solid #ddd')
    ]},
    {'selector': 'tr:nth-of-type(even)', 'props': [
        ('background-color', '#f9f9f9')
    ]},
    {'selector': 'tr:hover', 'props': [
        ('background-color', '#f5f5f5')
    ]}
])

print("\n📋 FORMATTED TABLE:")
display(styled_df)

# Print summary statistics
print("\n" + "=" * 80)
print("SUMMARY STATISTICS")
print("=" * 80)
total_2024 = df_summary['2024 Violations'].sum()
avg_growth_2019 = df_summary[df_summary['Growth 2019-2024 (%)'] != float('inf')]['Growth 2019-2024 (%)'].mean()
avg_growth_2016 = df_summary[df_summary['Growth 2016-2024 (%)'] != float('inf')]['Growth 2016-2024 (%)'].mean()

print(f"Total violations in top 20 (2024): {total_2024:,}")
print(f"Average growth (2019-2024): {avg_growth_2019:+.1f}%")
print(f"Average growth (2016-2024): {avg_growth_2016:+.1f}%")

print("\n" + "=" * 80)
print("✓ TABLE GENERATION COMPLETE")
print("=" * 80)

CREATING TOP 20 CRIMES TABLE

📊 TOP 20 ORGANIZED CRIMES IN CANADA (2024)
 Rank                                           Violation Type 2024 Violations Growth 2019-2024 (%) Growth 2016-2024 (%)
    1                                                    Fraud           6,282               +75.5%              +327.9%
    2                                      Motor Vehicle Theft           2,036              +770.1%              +874.2%
    3  Drug (other than cannabis) - trafficking and production           1,094                +4.8%              +127.4%
    4                                          Break and enter             712               +95.6%              +431.3%
    5                                                  Assault             511               +34.8%               +83.8%
    6                                                 Mischief             428               +98.1%              +169.2%
    7                                           Identity Theft             298  

,Rank,Violation Type,2024 Violations,Growth 2019-2024 (%),Growth 2016-2024 (%)
0,1,Fraud,"6,282",+75.5%,+327.9%
1,2,Motor Vehicle Theft,"2,036",+770.1%,+874.2%
2,3,Drug (other than cannabis) - trafficking and production,"1,094",+4.8%,+127.4%
3,4,Break and enter,712,+95.6%,+431.3%
4,5,Assault,511,+34.8%,+83.8%
5,6,Mischief,428,+98.1%,+169.2%
6,7,Identity Theft,298,+302.7%,+2880.0%
7,8,Robbery,286,+43.0%,+155.4%
8,9,"Shoplifting $5,000 or under",220,+91.3%,+254.8%
9,10,Extortion,209,+422.5%,+553.1%



SUMMARY STATISTICS
Total violations in top 20 (2024): 13,508
Average growth (2019-2024): +138.1%
Average growth (2016-2024): +372.8%

✓ TABLE GENERATION COMPLETE


In [ ]:
# Create Organized Crime Line Graph - Organized Crime Trends (2016-2024)
print("=" * 80)
print("CREATING LINE GRAPH VISUALIZATION")
print("=" * 80)

# Calculate total organized crime per year
yearly_totals = df_violations.groupby('Year')[value_col].sum().reset_index()
yearly_totals.columns = ['Year', 'Total_Violations']

print(f"\n📊 Yearly Totals:")
print(yearly_totals)

# Find the top 3 most common organized crime violations (by total across all years)
top_violations = df_violations.groupby(violation_col)[value_col].sum().nlargest(3)
print(f"\n🔝 Top 3 Most Common Organized Crime Violations (2016-2024):")
for i, (violation, total) in enumerate(top_violations.items(), 1):
    print(f"   {i}. {violation}: {total:,} total violations")

# Prepare data for the top 3 violations over time
top_3_names = top_violations.index.tolist()
df_top3 = df_violations[df_violations[violation_col].isin(top_3_names)]

# Pivot data for easier plotting
df_pivot = df_top3.pivot_table(
    index='Year',
    columns=violation_col,
    values=value_col,
    aggfunc='sum'
).fillna(0)

# Create the line graph
fig, ax = plt.subplots(figsize=(14, 8))

# Plot total organized crime
ax.plot(yearly_totals['Year'], yearly_totals['Total_Violations'],
        marker='o', linewidth=3, markersize=8,
        label='Total Organized Crime', color='#2E86AB', linestyle='-')

# Define colors for top 3 violations
colors = ['#A23B72', '#F18F01', '#C73E1D']
linestyles = ['--', '-.', ':']

# Plot top 3 violations
for i, violation in enumerate(top_3_names):
    if violation in df_pivot.columns:
        ax.plot(df_pivot.index, df_pivot[violation],
                marker='s', linewidth=2.5, markersize=6,
                label=violation, color=colors[i], linestyle=linestyles[i])

# Formatting
ax.set_xlabel('Year', fontsize=14, fontweight='bold')
ax.set_ylabel('Number of Violations', fontsize=14, fontweight='bold')
ax.set_title('Organized Crime Trends in Canada (2016-2024)\nTotal Organized Crime and Top 3 Violation Types',
             fontsize=16, fontweight='bold', pad=20)

# Format axes
ax.set_xticks(range(2016, 2025))
ax.grid(True, alpha=0.3, linestyle='--')
ax.legend(fontsize=11, loc='best', framealpha=0.9)

# Format y-axis with commas
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f'{int(x):,}'))

# Add subtle background
ax.set_facecolor('#F8F9FA')
fig.patch.set_facecolor('white')

plt.tight_layout()

# Save the organized crime figure
output_path = outputs_dir / 'organized_crime_trends_2016_2024.png'
plt.savefig(output_path, dpi=300, bbox_inches='tight')
print(f"\n✓ Graph saved to: {output_path}")

plt.show()

print("\n" + "=" * 80)
print("✓ LINE GRAPH COMPLETE")
print("=" * 80)

In [ ]:
# Fetch cybercrime data
print("=" * 80)
print("FETCHING CYBERCRIME DATA FROM STATISTICS CANADA")
print("=" * 80)

# StatCan table ID for cybercrime
CYBER_TABLE_ID = "3510000101"

# Construct the download URL (using the direct ZIP download link)
cyber_download_url = f"https://www150.statcan.gc.ca/n1/tbl/csv/{CYBER_TABLE_ID}-eng.zip"

try:
    print(f"\n📥 Downloading data from Statistics Canada (Table {CYBER_TABLE_ID})...")
    print(f"URL: {cyber_download_url[:80]}...")

    # Download the data
    response = requests.get(cyber_download_url, timeout=30)
    response.raise_for_status()

    # Extract ZIP file
    with zipfile.ZipFile(BytesIO(response.content)) as zip_file:
        # Get the CSV file name (should be the table ID)
        csv_files = [f for f in zip_file.namelist() if f.endswith('.csv')]

        if not csv_files:
            raise ValueError("No CSV file found in downloaded ZIP")

        csv_filename = csv_files[0]
        print(f"✓ Downloaded and extracted: {csv_filename}")

        # Read the CSV
        with zip_file.open(csv_filename) as csv_file:
            df_cyber = pd.read_csv(csv_file)

    print(f"✓ Data loaded successfully: {len(df_cyber):,} rows, {len(df_cyber.columns)} columns")

    # Save raw data
    raw_data_path = data_dir / 'cybercrime_raw.csv'
    df_cyber.to_csv(raw_data_path, index=False)
    print(f"✓ Raw data saved to: {raw_data_path}")

    # Display basic info
    print(f"\n📊 Dataset Overview:")
    print(f"   • Time period: {df_cyber['REF_DATE'].min()} to {df_cyber['REF_DATE'].max()}")
    print(f"   • Columns: {', '.join(df_cyber.columns.tolist())}")

except requests.exceptions.RequestException as e:
    print(f"❌ Error downloading data: {e}")
    raise
except Exception as e:
    print(f"❌ Error processing data: {e}")
    raise

print("\n" + "=" * 80)
print("✓ DATA FETCH COMPLETE")
print("=" * 80)

In [ ]:
# Cybercrime - Process and Clean Data
print("=" * 80)
print("PROCESSING CYBERCRIME DATA")
print("=" * 80)

# Display the structure
print(f"\n📋 Data Structure:")
print(df_cyber.head())
print(f"\n📊 Column Types:")
print(df_cyber.dtypes)

# Clean and prepare the data
df_cyber_clean = df_cyber.copy()

# Convert REF_DATE to integer year
df_cyber_clean['Year'] = df_cyber_clean['REF_DATE'].astype(int)

# Get the value column (usually 'VALUE')
cyber_value_col = 'VALUE'
if cyber_value_col not in df_cyber_clean.columns:
    possible_cols = [col for col in df_cyber_clean.columns if 'value' in col.lower()]
    if possible_cols:
        cyber_value_col = possible_cols[0]
    else:
        raise ValueError("Could not find value column in data")

print(f"\n✓ Using '{cyber_value_col}' as the value column")

# Remove any rows with missing values
df_cyber_clean = df_cyber_clean[df_cyber_clean[cyber_value_col].notna()]

# Get violation column name - for cybercrime, it might be 'Violations' or something similar
cyber_violation_col = 'Violations'
if cyber_violation_col not in df_cyber_clean.columns:
    possible_cols = [col for col in df_cyber_clean.columns if 'violation' in col.lower() or 'type' in col.lower() or 'category' in col.lower()]
    if possible_cols:
        cyber_violation_col = possible_cols[0]
    else:
        # If we can't find a violation column, list all columns to debug
        print(f"Available columns: {df_cyber_clean.columns.tolist()}")
        raise ValueError("Could not find violation/type column in data")

print(f"✓ Using '{cyber_violation_col}' as the violation column")

# Display unique violations/types
print(f"\n🔍 Unique violation types found: {df_cyber_clean[cyber_violation_col].nunique()}")
print("\nSample violations:")
for i, violation in enumerate(df_cyber_clean[cyber_violation_col].unique()[:10], 1):
    print(f"   {i}. {violation}")

# Filter out "Total" rows to get specific violations
df_cyber_violations = df_cyber_clean[~df_cyber_clean[cyber_violation_col].str.contains('Total', case=False, na=False)]

print(f"\n✓ Filtered to {len(df_cyber_violations):,} rows of specific violations")
print(f"✓ Years covered: {df_cyber_violations['Year'].min()} to {df_cyber_violations['Year'].max()}")
print(f"✓ Number of specific violation types: {df_cyber_violations[cyber_violation_col].nunique()}")

print("\n" + "=" * 80)
print("✓ DATA PROCESSING COMPLETE")
print("=" * 80)

In [ ]:
# Create Cybercrime Table - Top 20 Crimes in 2024 with Growth Metrics
print("=" * 80)
print("CREATING TOP 20 CYBERCR IMES TABLE")
print("=" * 80)

# Determine the most recent year available in the data
max_year = df_cyber_violations['Year'].max()
print(f"\n📅 Most recent year in dataset: {max_year}")

# Get most recent year data
df_cyber_latest = df_cyber_violations[df_cyber_violations['Year'] == max_year].copy()

# Try to get comparison years (5 and 8 years back if available)
available_years = sorted(df_cyber_violations['Year'].unique())
print(f"📅 Available years: {available_years}")

# Find appropriate comparison years
if max_year - 5 in available_years:
    compare_year_1 = max_year - 5
else:
    compare_year_1 = available_years[len(available_years)//2] if len(available_years) > 1 else available_years[0]

if max_year - 8 in available_years:
    compare_year_2 = max_year - 8
else:
    compare_year_2 = available_years[0]

print(f"📅 Comparison years: {compare_year_2} and {compare_year_1}")

df_cyber_comp1 = df_cyber_violations[df_cyber_violations['Year'] == compare_year_1].copy()
df_cyber_comp2 = df_cyber_violations[df_cyber_violations['Year'] == compare_year_2].copy()

# Calculate totals for each year by violation type
cyber_latest = df_cyber_latest.groupby(cyber_violation_col)[cyber_value_col].sum()
cyber_comp1 = df_cyber_comp1.groupby(cyber_violation_col)[cyber_value_col].sum()
cyber_comp2 = df_cyber_comp2.groupby(cyber_violation_col)[cyber_value_col].sum()

# Get top 20 crimes by most recent year violations
top_20_cyber = cyber_latest.nlargest(20)

# Create the summary table
cyber_summary_data = []

for violation in top_20_cyber.index:
    count_latest = cyber_latest.get(violation, 0)
    count_comp1 = cyber_comp1.get(violation, 0)
    count_comp2 = cyber_comp2.get(violation, 0)

    # Calculate growth
    if count_comp1 > 0:
        growth_comp1 = ((count_latest - count_comp1) / count_comp1) * 100
    else:
        growth_comp1 = float('inf') if count_latest > 0 else 0

    if count_comp2 > 0:
        growth_comp2 = ((count_latest - count_comp2) / count_comp2) * 100
    else:
        growth_comp2 = float('inf') if count_latest > 0 else 0

    cyber_summary_data.append({
        'Rank': len(cyber_summary_data) + 1,
        'Violation Type': violation,
        f'{max_year} Violations': int(count_latest),
        f'Growth {compare_year_1}-{max_year} (%)': growth_comp1,
        f'Growth {compare_year_2}-{max_year} (%)': growth_comp2
    })

# Create DataFrame
df_cyber_summary = pd.DataFrame(cyber_summary_data)

# Format the table for display
df_cyber_display = df_cyber_summary.copy()
df_cyber_display[f'{max_year} Violations'] = df_cyber_display[f'{max_year} Violations'].apply(lambda x: f'{x:,}')
df_cyber_display[f'Growth {compare_year_1}-{max_year} (%)'] = df_cyber_display[f'Growth {compare_year_1}-{max_year} (%)'].apply(
    lambda x: 'N/A' if x == float('inf') else f'{x:+.1f}%'
)
df_cyber_display[f'Growth {compare_year_2}-{max_year} (%)'] = df_cyber_display[f'Growth {compare_year_2}-{max_year} (%)'].apply(
    lambda x: 'N/A' if x == float('inf') else f'{x:+.1f}%'
)

print(f"\n📊 TOP 20 CYBERCRIMES IN CANADA ({max_year})")
print("=" * 80)
print(df_cyber_display.to_string(index=False))
print("=" * 80)

# Save to CSV
csv_path = outputs_dir / f'top_20_cybercrimes_{max_year}_with_growth.csv'
df_cyber_summary.to_csv(csv_path, index=False)
print(f"\n✓ Table saved to: {csv_path}")

# Create a nicely formatted display table
from IPython.display import display, HTML

# Style the dataframe for better display
styled_cyber_df = df_cyber_display.style.set_properties(**{
    'text-align': 'left',
    'font-size': '12px',
    'border-color': 'black'
}).set_table_styles([
    {'selector': 'th', 'props': [
        ('background-color', '#9333EA'),
        ('color', 'white'),
        ('font-weight', 'bold'),
        ('text-align', 'center'),
        ('padding', '10px')
    ]},
    {'selector': 'td', 'props': [
        ('padding', '8px'),
        ('border', '1px solid #ddd')
    ]},
    {'selector': 'tr:nth-of-type(even)', 'props': [
        ('background-color', '#f9f9f9')
    ]},
    {'selector': 'tr:hover', 'props': [
        ('background-color', '#f5f5f5')
    ]}
])

print("\n📋 FORMATTED TABLE:")
display(styled_cyber_df)

# Print summary statistics
print("\n" + "=" * 80)
print("SUMMARY STATISTICS")
print("=" * 80)
total_latest = df_cyber_summary[f'{max_year} Violations'].sum()
avg_growth_comp1 = df_cyber_summary[df_cyber_summary[f'Growth {compare_year_1}-{max_year} (%)'] != float('inf')][f'Growth {compare_year_1}-{max_year} (%)'].mean()
avg_growth_comp2 = df_cyber_summary[df_cyber_summary[f'Growth {compare_year_2}-{max_year} (%)'] != float('inf')][f'Growth {compare_year_2}-{max_year} (%)'].mean()

print(f"Total violations in top 20 ({max_year}): {total_latest:,}")
print(f"Average growth ({compare_year_1}-{max_year}): {avg_growth_comp1:+.1f}%")
print(f"Average growth ({compare_year_2}-{max_year}): {avg_growth_comp2:+.1f}%")

print("\n" + "=" * 80)
print("✓ TABLE GENERATION COMPLETE")
print("=" * 80)

In [ ]:
# Create Cybercrime Line Graph - Cybercrime Trends
print("=" * 80)
print("CREATING CYBERCRIME LINE GRAPH VISUALIZATION")
print("=" * 80)

# Calculate total cybercrime per year
cyber_yearly_totals = df_cyber_violations.groupby('Year')[cyber_value_col].sum().reset_index()
cyber_yearly_totals.columns = ['Year', 'Total_Violations']

print(f"\n📊 Yearly Totals:")
print(cyber_yearly_totals)

# Find the top 3 most common cybercrime violations (by total across all years)
top_cyber_violations = df_cyber_violations.groupby(cyber_violation_col)[cyber_value_col].sum().nlargest(3)
print(f"\n🔝 Top 3 Most Common Cybercrime Violations:")
for i, (violation, total) in enumerate(top_cyber_violations.items(), 1):
    print(f"   {i}. {violation}: {total:,} total violations")

# Prepare data for the top 3 violations over time
top_3_cyber_names = top_cyber_violations.index.tolist()
df_cyber_top3 = df_cyber_violations[df_cyber_violations[cyber_violation_col].isin(top_3_cyber_names)]

# Pivot data for easier plotting
df_cyber_pivot = df_cyber_top3.pivot_table(
    index='Year',
    columns=cyber_violation_col,
    values=cyber_value_col,
    aggfunc='sum'
).fillna(0)

# Create the line graph
fig, ax = plt.subplots(figsize=(14, 8))

# Plot total cybercrime
ax.plot(cyber_yearly_totals['Year'], cyber_yearly_totals['Total_Violations'],
        marker='o', linewidth=3, markersize=8,
        label='Total Cybercrime', color='#9333EA', linestyle='-')

# Define colors for top 3 violations (purple theme for cyber)
cyber_colors = ['#EC4899', '#8B5CF6', '#06B6D4']
cyber_linestyles = ['--', '-.', ':']

# Plot top 3 violations
for i, violation in enumerate(top_3_cyber_names):
    if violation in df_cyber_pivot.columns:
        ax.plot(df_cyber_pivot.index, df_cyber_pivot[violation],
                marker='s', linewidth=2.5, markersize=6,
                label=violation, color=cyber_colors[i], linestyle=cyber_linestyles[i])

# Formatting
ax.set_xlabel('Year', fontsize=14, fontweight='bold')
ax.set_ylabel('Number of Violations', fontsize=14, fontweight='bold')

# Determine year range for title
min_year = cyber_yearly_totals['Year'].min()
max_year = cyber_yearly_totals['Year'].max()
ax.set_title(f'Cybercrime Trends in Canada ({min_year}-{max_year})\nTotal Cybercrime and Top 3 Violation Types',
             fontsize=16, fontweight='bold', pad=20)

# Format axes - dynamic year range
year_range = list(range(int(min_year), int(max_year) + 1))
ax.set_xticks(year_range)
ax.grid(True, alpha=0.3, linestyle='--')
ax.legend(fontsize=11, loc='best', framealpha=0.9)

# Format y-axis with commas
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f'{int(x):,}'))

# Add subtle background
ax.set_facecolor('#FAF5FF')  # Light purple background
fig.patch.set_facecolor('white')

plt.tight_layout()

# Save the cybercrime figure
output_path = outputs_dir / f'cybercrime_trends_{min_year}_{max_year}.png'
plt.savefig(output_path, dpi=300, bbox_inches='tight')
print(f"\n✓ Graph saved to: {output_path}")

plt.show()

print("\n" + "=" * 80)
print("✓ LINE GRAPH COMPLETE")
print("=" * 80)

In [ ]:
# CSI: Fetch Data

# Define the table PID
pid = '35100026'

# Fetch the download URL from the API
api_url = f"https://www150.statcan.gc.ca/t1/wds/rest/getFullTableDownloadCSV/{pid}/en"
response = requests.get(api_url)

if not response.ok:
    raise ValueError(f"API request failed: {response.status_code} - {response.text}")

zip_url = response.json()['object']

# Download the ZIP file
zip_response = requests.get(zip_url)

if not zip_response.ok:
    raise ValueError(f"Data download failed: {zip_response.status_code} - {zip_response.text}")

# Extract and load the CSV
with zipfile.ZipFile(BytesIO(zip_response.content)) as z:
    csv_filename = f"{pid.replace('-', '')}.csv"
    if csv_filename not in z.namelist():
        raise ValueError("CSV file not found in ZIP")
    df = pd.read_csv(z.open(csv_filename), low_memory=False)

print("Data loaded successfully.")
print(f"Shape: {df.shape}")
print(f"Columns: {df.columns.tolist()}")

Data loaded successfully.
Shape: (22908, 15)
Columns: ['REF_DATE', 'GEO', 'DGUID', 'Statistics', 'UOM', 'UOM_ID', 'SCALAR_FACTOR', 'SCALAR_ID', 'VECTOR', 'COORDINATE', 'VALUE', 'STATUS', 'SYMBOL', 'TERMINATED', 'DECIMALS']


In [ ]:
# ============================================================
# CSI: Filter and Process Data
# ============================================================

# Filter for Canada and relevant indices
df_filtered = df[(df['GEO'] == 'Canada') &
                 (df['Statistics'].isin(['Crime severity index',
                                                   'Violent crime severity index',
                                                   'Non-violent crime severity index']))]

# Pivot the data for plotting
pivot_df = df_filtered.pivot(index='REF_DATE',
                              columns='Statistics',
                              values='VALUE').reset_index()

# Ensure data covers 1998-2024
available_years = pivot_df['REF_DATE'].unique()
print(f"Available years: {sorted(available_years)}")

if not all(year in available_years for year in range(1998, 2025)):
    print("Warning: Data does not cover the full range from 1998 to 2024")
    print(f"Missing years: {set(range(1998, 2025)) - set(available_years)}")

print(f"\nData preview:")
print(pivot_df.head())
print(f"\nData shape: {pivot_df.shape}")

Available years: [np.int64(1998), np.int64(1999), np.int64(2000), np.int64(2001), np.int64(2002), np.int64(2003), np.int64(2004), np.int64(2005), np.int64(2006), np.int64(2007), np.int64(2008), np.int64(2009), np.int64(2010), np.int64(2011), np.int64(2012), np.int64(2013), np.int64(2014), np.int64(2015), np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024)]

Data preview:
Statistics  REF_DATE  Crime severity index  Non-violent crime severity index  \
0               1998                118.84                            126.93   
1               1999                111.24                            115.80   
2               2000                106.73                            110.17   
3               2001                105.30                            108.41   
4               2002                104.14                            107.19   

Statistics  Violent crime severity index  
0          

In [ ]:

# ============================================================
# CSI: Create Visualization
# ============================================================

# Plot the line graph
plt.figure(figsize=(12, 6))
plt.plot(pivot_df['REF_DATE'], pivot_df['Crime severity index'],
         label='Overall Crime Severity Index', marker='o', linewidth=2)
plt.plot(pivot_df['REF_DATE'], pivot_df['Violent crime severity index'],
         label='Violent Crime Severity Index', marker='o', linewidth=2)
plt.plot(pivot_df['REF_DATE'], pivot_df['Non-violent crime severity index'],
         label='Non-Violent Crime Severity Index', marker='o', linewidth=2)

plt.title('Crime Severity Index Trends in Canada (1998-2024)', fontsize=14, fontweight='bold')
plt.xlabel('Year', fontsize=12)
plt.ylabel('Index Value', fontsize=12)
plt.grid(True, alpha=0.3)
plt.legend(fontsize=10)
plt.xticks(range(1998, 2025, 2), rotation=45)
plt.tight_layout()

# Save the CSI figure
plt.savefig('outputs/csi_trend.png', dpi=300, bbox_inches='tight')
print("✓ CSI chart saved to outputs/csi_trend.png")

plt.show()



In [ ]:

# ============================================================
# CSI: Calculate Growth Rates
# ============================================================

# Extract values for 2019 and 2024
df_2019 = pivot_df[pivot_df['REF_DATE'] == 2019].set_index('REF_DATE')
df_2024 = pivot_df[pivot_df['REF_DATE'] == 2024].set_index('REF_DATE')

if df_2019.empty:
    print("Warning: No data available for 2019")
if df_2024.empty:
    print("Warning: No data available for 2024")

if not df_2019.empty and not df_2024.empty:
    # Calculate growth rates using iloc[0] to access the single row of values
    growth_rates = ((df_2024.iloc[0] - df_2019.iloc[0]) / df_2019.iloc[0] * 100).round(2)

    # Convert the resulting Series to a DataFrame with the desired column name
    growth_rates = growth_rates.to_frame(name='Growth Rate 2019-2024 (%)')

    # Display as a table
    print("\nGrowth Rates from 2019 to 2024:")
    print(growth_rates)
    print("\n" + "="*50)

    # Also display the actual values for context
    print("\n2019 Values:")
    print(df_2019.T)
    print("\n2024 Values:")
    print(df_2024.T)
else:
    print("Cannot calculate growth rates - missing data for 2019 or 2024")


Growth Rates from 2019 to 2024:
                                  Growth Rate 2019-2024 (%)
Statistics                                                 
Crime severity index                                  -2.30
Non-violent crime severity index                      -7.91
Violent crime severity index                          10.67


2019 Values:
REF_DATE                           2019
Statistics                             
Crime severity index              79.72
Non-violent crime severity index  75.75
Violent crime severity index      90.24

2024 Values:
REF_DATE                           2024
Statistics                             
Crime severity index              77.89
Non-violent crime severity index  69.76
Violent crime severity index      99.87


In [ ]:
# ============================================================
# FULL HTML REPORT GENERATOR (WITH CYBERCRIME)
# ============================================================

def generate_full_html_report():
    """
    Generate full HTML report with 3 datasets:
    - Crime Severity Index
    - Organized Crime
    - Cybercrime
    """

    # Determine cybercrime year range dynamically
    cyber_min_year = df_cyber_violations['Year'].min()
    cyber_max_year = df_cyber_violations['Year'].max()

    html_content = f"""<!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>Canada Crime Statistics Analysis - Full Report</title>
    <style>
        * {{
            margin: 0;
            padding: 0;
            box-sizing: border-box;
        }}

        body {{
            font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', 'Roboto', 'Oxygen', 'Ubuntu', sans-serif;
            line-height: 1.6;
            color: #2d3748;
            background: #f7fafc;
            padding: 20px;
        }}

        .container {{
            max-width: 1200px;
            margin: 0 auto;
            background: white;
            box-shadow: 0 4px 6px rgba(0, 0, 0, 0.1);
            border-radius: 8px;
            overflow: hidden;
        }}

        .header {{
            background: linear-gradient(135deg, #667eea 0%, #764ba2 100%);
            color: white;
            padding: 40px;
            text-align: center;
        }}

        .header h1 {{
            font-size: 2.5em;
            margin-bottom: 10px;
        }}

        .header .subtitle {{
            font-size: 1.2em;
            opacity: 0.9;
        }}

        .full-release-badge {{
            background: rgba(255, 255, 255, 0.3);
            display: inline-block;
            padding: 8px 20px;
            border-radius: 20px;
            margin: 15px 0;
            font-weight: 600;
        }}

        .content {{
            padding: 40px;
        }}

        .section {{
            margin-bottom: 50px;
        }}

        .section-title {{
            font-size: 2em;
            color: #2d3748;
            margin-bottom: 20px;
            padding-bottom: 10px;
            border-bottom: 3px solid #667eea;
        }}

        .chart-container {{
            margin: 30px 0;
            text-align: center;
        }}

        .chart-container img {{
            max-width: 100%;
            height: auto;
            border: 1px solid #e2e8f0;
            border-radius: 8px;
            box-shadow: 0 2px 4px rgba(0,0,0,0.1);
        }}

        .summary-box {{
            background: #edf2f7;
            padding: 20px;
            border-left: 4px solid #667eea;
            border-radius: 4px;
            margin: 20px 0;
        }}

        .footer {{
            background: #2d3748;
            color: white;
            padding: 30px;
            text-align: center;
        }}

        @media (max-width: 768px) {{
            .header h1 {{
                font-size: 1.8em;
            }}

            .content {{
                padding: 20px;
            }}
        }}
    </style>
</head>
<body>
    <div class="container">
        <!-- Header -->
        <div class="header">
            <h1>🇨🇦 Crime in Canada</h1>
            <div class="full-release-badge">FULL REPORT - 3 Core Datasets</div>
            <p class="subtitle">Evidence-Based Analysis from Statistics Canada</p>
            <p style="margin-top: 15px; opacity: 0.8;">Generated: {datetime.now().strftime('%B %d, %Y')}</p>
            <p style="margin-top: 5px; opacity: 0.7; font-size: 0.9em;">Author: tonHS</p>
        </div>

        <!-- Content -->
        <div class="content">
            <!-- Executive Summary -->
            <div class="section">
                <h2 class="section-title">📊 Executive Summary</h2>
                <div class="summary-box">
                    <p><strong>This report analyzes three core crime datasets from Statistics Canada:</strong></p>
                    <ol style="margin-top: 15px; margin-left: 20px; line-height: 2;">
                        <li><strong>Crime Severity Index (1998-2024):</strong> Comprehensive measure of crime severity across Canada</li>
                        <li><strong>Organized Crime (2016-2024):</strong> Analysis of organized crime violations and trends</li>
                        <li><strong>Cybercrime ({cyber_min_year}-{cyber_max_year}):</strong> Digital and online crime violations</li>
                    </ol>
                </div>
            </div>

            <!-- Section 1: Crime Severity Index -->
            <div class="section">
                <h2 class="section-title">📈 1. Crime Severity Index</h2>
                <p style="font-size: 1.1em; line-height: 1.8;">The Crime Severity Index (CSI) measures both the volume and severity of crime in Canada.</p>

                <div class="chart-container">
                    <h3 style="margin-bottom: 15px; color: #4a5568;">Crime Severity Trends (1998-2024)</h3>
                    <img src="csi_trend.png" alt="Crime Severity Index Trends">
                    <p style="margin-top: 10px; font-size: 0.9em; color: #718096;">Source: Statistics Canada Table 35-10-0026-01</p>
                </div>
            </div>

            <!-- Section 2: Organized Crime -->
            <div class="section">
                <h2 class="section-title">🎯 2. Organized Crime</h2>
                <p style="font-size: 1.1em; line-height: 1.8;">Analysis of organized crime violations in Canada from 2016 to 2024.</p>

                <div class="chart-container">
                    <h3 style="margin-bottom: 15px; color: #4a5568;">Organized Crime Trends (2016-2024)</h3>
                    <img src="organized_crime_trends_2016_2024.png" alt="Organized Crime Trends">
                    <p style="margin-top: 10px; font-size: 0.9em; color: #718096;">Source: Statistics Canada Table 35-10-0062-01</p>
                </div>
            </div>

            <!-- Section 3: Cybercrime -->
            <div class="section">
                <h2 class="section-title">💻 3. Cybercrime</h2>
                <p style="font-size: 1.1em; line-height: 1.8;">Analysis of cybercrime violations in Canada from {cyber_min_year} to {cyber_max_year}, tracking digital and online criminal activity.</p>

                <div class="chart-container">
                    <h3 style="margin-bottom: 15px; color: #4a5568;">Cybercrime Trends ({cyber_min_year}-{cyber_max_year})</h3>
                    <img src="cybercrime_trends_{cyber_min_year}_{cyber_max_year}.png" alt="Cybercrime Trends">
                    <p style="margin-top: 10px; font-size: 0.9em; color: #718096;">Source: Statistics Canada Table 35-10-0001-01</p>
                </div>

                <h3 style="margin-top: 30px; margin-bottom: 15px; color: #4a5568;">Key Findings:</h3>
                <ul style="margin-left: 30px; line-height: 2;">
                    <li><strong>Total cybercrime violations</strong> tracked from {cyber_min_year}-{cyber_max_year}</li>
                    <li><strong>Top 3 most common cybercrime categories</strong> identified and analyzed</li>
                    <li>Trends reveal patterns in digital criminal activity</li>
                </ul>

                <div style="background: #FAF5FF; padding: 20px; border-radius: 8px; margin-top: 25px; border-left: 4px solid #9333EA;">
                    <h4 style="color: #2d3748; margin-bottom: 10px;">About Cybercrime Data</h4>
                    <p style="line-height: 1.8;">This dataset tracks cyber-related violations including fraud, identity theft, and other digital crimes reported to law enforcement.</p>
                </div>
            </div>

            <!-- Methodology -->
            <div class="section">
                <h2 class="section-title">📖 Methodology</h2>

                <h3 style="color: #4a5568; margin-bottom: 10px;">Data Sources</h3>
                <p>All data is sourced from Statistics Canada's official public tables:</p>
                <ul style="margin: 15px 0 15px 30px; line-height: 2;">
                    <li><strong>Crime Severity Index:</strong> Table 35-10-0026-01</li>
                    <li><strong>Organized Crime:</strong> Table 35-10-0062-01</li>
                    <li><strong>Cybercrime:</strong> Table 35-10-0001-01</li>
                </ul>

                <h3 style="margin-top: 25px; color: #4a5568; margin-bottom: 10px;">Limitations</h3>
                <ul style="margin: 15px 0 15px 30px; line-height: 2;">
                    <li><strong>Reported crimes only:</strong> Data reflects crimes reported to police</li>
                    <li><strong>Geographic scope:</strong> Canada-wide data</li>
                </ul>
            </div>
        </div>

        <!-- Footer -->
        <div class="footer">
            <p style="font-size: 1.2em;"><strong>Canada Crime Statistics Analysis</strong></p>
            <p style="margin-top: 15px; opacity: 0.9;">Data Source: Statistics Canada | Analysis: tonHS</p>
            <p style="margin-top: 10px; opacity: 0.8;">Generated {datetime.now().strftime('%B %d, %Y')}</p>
        </div>
    </div>
</body>
</html>"""

    return html_content

# Generate the HTML
print("=" * 80)
print("GENERATING FULL HTML REPORT WITH CYBERCRIME")
print("=" * 80)

html_report = generate_full_html_report()

# Save the HTML file
html_path = outputs_dir / 'index.html'
with open(html_path, 'w', encoding='utf-8') as f:
    f.write(html_report)

print(f"\n✓ HTML report generated successfully!")
print(f"✓ Saved to: {html_path}")

# List all files in outputs directory
print(f"\n📁 Files in outputs directory:")
for file in sorted(outputs_dir.iterdir()):
    file_size = file.stat().st_size
    if file_size < 1024:
        size_str = f"{file_size} bytes"
    elif file_size < 1024*1024:
        size_str = f"{file_size/1024:.1f} KB"
    else:
        size_str = f"{file_size/(1024*1024):.1f} MB"
    print(f"   • {file.name} ({size_str})")

print("\n" + "=" * 80)
print("✓ HTML GENERATION COMPLETE")
print("=" * 80)
print("\n📋 NEXT STEPS:")
print("1. Check that you have these files:")
print("   - index.html")
print("   - csi_trend.png")
print("   - organized_crime_trends_2016_2024.png")
print(f"   - cybercrime_trends_{{year}}_{{year}}.png")
print("\n2. All visualizations and data are ready for GitHub Pages!")